#  Titanic Dataset — Complete Data Analysis

**Steps Covered:** 1 (Explore) → 2 (Clean & Transform) → 3 (EDA & Visualize) → 7 (Document) → 8 (Git)

---
**Dataset:** 891 Titanic passengers  
**Target:** `survived` (0 = Died, 1 = Survived)  
**Libraries:** Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Visual theme
PALETTE = ['#2E86AB', '#E84855', '#3BB273', '#F4A261', '#9B5DE5', '#F15BB5']
BG = '#F8F9FB'
sns.set_theme(style='whitegrid', palette=PALETTE)
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG,
    'axes.titlesize': 13, 'axes.titleweight': 'bold'
})
print(' Libraries loaded successfully')

---
## Step 1 — Data Collection & Exploration
### 1a. Dataset Loading

In [ ]:
# Load the raw Titanic dataset into a Pandas DataFrame
df_raw = pd.read_csv('../data/titanic_raw.csv')
df = df_raw.copy()  # Preserve the original — always work on a copy

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

### 1b. Structure Examination

In [ ]:
# Column-level info: dtype, non-null counts, nulls, unique values
info_df = pd.DataFrame({
    'Dtype'   : df.dtypes,
    'Non-Null': df.notnull().sum(),
    'Null'    : df.isnull().sum(),
    'Unique'  : df.nunique()
})
info_df

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Last 5 rows
df.tail()

### 1c. Summary Statistics

In [ ]:
# Full numerical summary
df.describe(include=[np.number]).round(3)

In [ ]:
# Categorical value counts
for col in df.select_dtypes(include='object').columns:
    print(f'\n── {col} ──')
    print(df[col].value_counts(dropna=False))

### 1d. Problem Identification

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
mv = pd.DataFrame({'Missing': missing, 'Pct %': missing_pct})
mv[mv['Missing'] > 0].sort_values('Pct %', ascending=False)

In [ ]:
# Duplicate rows
print(f'Duplicate rows: {df.duplicated().sum()}')

# IQR-based outlier detection
print('\nOutlier Detection (IQR method):')
for col in ['age', 'fare', 'sibsp', 'parch']:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f'  {col:8s}: bounds [{lo:.1f}, {hi:.1f}]  → {n_out} outliers')

---
## Step 2 — Data Cleaning & Transformation
### 2a. Handle Missing Values

In [ ]:
# Age: grouped median imputation (pclass × sex) — smarter than global median
df['age'] = df.groupby(['pclass', 'sex'])['age'].transform(
    lambda x: x.fillna(x.median())
)

# Embarked: mode imputation for the few missing values
if df['embarked'].isnull().sum() > 0:
    df['embarked']    = df['embarked'].fillna(df['embarked'].mode()[0])
    df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode()[0])

# Deck: drop column — 76.5% missing, imputation meaningless
df.drop(columns=['deck'], inplace=True)

print(f'Missing values remaining: {df.isnull().sum().sum()} ')

### 2b. Outlier Detection & Treatment

In [ ]:
# Visualise fare outliers before treatment
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot(df['fare'].dropna(), patch_artist=True,
                boxprops=dict(facecolor=PALETTE[3], alpha=0.7),
                medianprops=dict(color='white', linewidth=2),
                flierprops=dict(marker='o', markerfacecolor=PALETTE[1], markersize=4))
axes[0].set_title('Fare — BEFORE Capping')

# Cap fare at 99th percentile
fare_p99 = df['fare'].quantile(0.99)
df['fare_raw'] = df['fare'].copy()
df['fare'] = df['fare'].clip(upper=fare_p99)

axes[1].boxplot(df['fare'].dropna(), patch_artist=True,
                boxprops=dict(facecolor=PALETTE[2], alpha=0.7),
                medianprops=dict(color='white', linewidth=2),
                flierprops=dict(marker='o', markerfacecolor=PALETTE[1], markersize=4))
axes[1].set_title(f'Fare — AFTER Capping at £{fare_p99:.0f}')
plt.tight_layout()
plt.show()
print(f'Outliers capped: {(df["fare_raw"] > fare_p99).sum()} values')

### 2c. Categorical Encoding

In [ ]:
# Label Encoding: binary categories
le = LabelEncoder()
df['sex_encoded']      = le.fit_transform(df['sex'])       # female=0, male=1
df['embarked_encoded'] = le.fit_transform(df['embarked'])  # C=0, Q=1, S=2

# One-Hot Encoding: pclass (no ordinal assumption)
df = pd.get_dummies(df, columns=['pclass'], prefix='pclass', drop_first=False)
for col in ['pclass_1','pclass_2','pclass_3']:
    if col in df.columns:
        df[col] = df[col].astype(int)

print('Encoding complete')
print(df[['sex', 'sex_encoded', 'embarked', 'embarked_encoded',
          'pclass_1', 'pclass_2', 'pclass_3']].head(6))

### 2d. Numerical Scaling (MinMax)

In [ ]:
# MinMax scale numeric features to [0, 1]
scaler = MinMaxScaler()
scale_cols = ['age', 'fare', 'sibsp', 'parch']
scaled = scaler.fit_transform(df[scale_cols])
for i, col in enumerate(scale_cols):
    df[f'{col}_scaled'] = scaled[:, i]

df[['age', 'age_scaled', 'fare', 'fare_scaled']].head()

### 2e. Feature Engineering

In [ ]:
# 1. Family size (total group on board)
df['family_size']    = df['sibsp'] + df['parch'] + 1

# 2. Is alone?
df['is_alone']       = (df['family_size'] == 1).astype(int)

# 3. Age group (non-linear binning)
df['age_group']      = pd.cut(df['age'], bins=[0,12,18,35,60,100],
                               labels=['Child','Teen','Young Adult','Adult','Senior'])

# 4. Title (social status proxy from who column)
df['title']          = df['who'].map({'man':'Mr','woman':'Mrs/Miss','child':'Master'})

# 5. Fare per person (adjusted cost)
df['fare_per_person'] = df['fare'] / df['family_size']

# 6. Fare band (wealth quartiles)
df['fare_band']      = pd.qcut(df['fare'], q=4,
                                labels=['Budget','Economy','Business','Premium'])

print(f'New shape: {df.shape}')
df[['family_size','is_alone','age_group','title','fare_per_person','fare_band']].head(8)

In [ ]:
# Save cleaned dataset
df.to_csv('../data/titanic_cleaned.csv', index=False)
print(' Cleaned dataset saved → data/titanic_cleaned.csv')
print(f'   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

---
## Step 3 — Exploratory Data Analysis
### 3a. Descriptive Statistics

In [ ]:
# Numerical descriptive statistics
df[['age','fare','sibsp','parch','family_size','fare_per_person']].describe().round(3)

In [ ]:
# Survival rates by all key categorical groups
for col in ['sex','age_group','title','fare_band','is_alone','embarked']:
    sr = df.groupby(col, observed=True)['survived'].agg(['mean','count'])
    sr.columns = ['Survival Rate %', 'N']
    sr['Survival Rate %'] = (sr['Survival Rate %'] * 100).round(1)
    print(f'\n── {col} ──')
    print(sr)

### 3b. Distribution Plots — Histograms & KDE Density Plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor=BG)
fig.suptitle('Distribution Plots — Histograms + KDE', fontsize=15, fontweight='bold')

# Age histogram + KDE overlay
df['age'].hist(bins=30, ax=axes[0,0], color=PALETTE[0], edgecolor='white', alpha=0.8)
ax2 = axes[0,0].twinx()
df['age'].plot.kde(ax=ax2, color=PALETTE[1], linewidth=2.5)
axes[0,0].axvline(df['age'].mean(), color='red', linestyle='--', lw=2, label=f'Mean {df["age"].mean():.1f}')
axes[0,0].set_title('Age Distribution')
axes[0,0].set_xlabel('Age')
axes[0,0].legend(fontsize=8)

# Fare histogram + KDE
df['fare'].hist(bins=30, ax=axes[0,1], color=PALETTE[3], edgecolor='white', alpha=0.8)
ax3 = axes[0,1].twinx()
df['fare'].plot.kde(ax=ax3, color=PALETTE[1], linewidth=2.5)
axes[0,1].set_title('Fare (£) Distribution')
axes[0,1].set_xlabel('Fare (£)')

# Family size
fs = df['family_size'].value_counts().sort_index()
axes[0,2].bar(fs.index, fs.values, color=PALETTE[4], edgecolor='white')
axes[0,2].set_title('Family Size Distribution')
axes[0,2].set_xlabel('Family Size')

# Fare per person — KDE density
df['fare_per_person'].plot.kde(ax=axes[1,0], color=PALETTE[5], linewidth=2.5)
axes[1,0].fill_between(axes[1,0].lines[0].get_xdata(), axes[1,0].lines[0].get_ydata(), alpha=0.15, color=PALETTE[5])
axes[1,0].set_title('Fare Per Person — KDE Density')
axes[1,0].set_xlabel('Fare Per Person (£)')
axes[1,0].set_xlim(0, df['fare_per_person'].quantile(0.98))

# Sex distribution
sex_c = df['sex'].value_counts()
axes[1,1].bar(sex_c.index.str.capitalize(), sex_c.values, color=[PALETTE[4],PALETTE[0]], edgecolor='white')
axes[1,1].set_title('Sex Distribution')

# Embarkation port — pie chart
emb_c = df['embark_town'].value_counts()
axes[1,2].pie(emb_c.values, labels=emb_c.index, autopct='%1.1f%%',
              colors=PALETTE[:3], wedgeprops={'edgecolor':'white','linewidth':2})
axes[1,2].set_title('Embarkation Port')

plt.tight_layout()
plt.show()

### 3c. Relationship Plots — Scatter, Grouped Bars, KDE Overlay

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor=BG)
fig.suptitle('Relationship Plots', fontsize=15, fontweight='bold')

# Scatter: Age vs Fare coloured by survival
sc = axes[0,0].scatter(df['age'], df['fare'], c=df['survived'],
    cmap='RdYlGn', alpha=0.5, edgecolors='white', linewidths=0.3, s=40)
plt.colorbar(sc, ax=axes[0,0], label='Survived')
axes[0,0].set_title('Age vs Fare (coloured by Survival)')
axes[0,0].set_xlabel('Age'); axes[0,0].set_ylabel('Fare (£)')

# Age vs Fare coloured by class
cls_colors = {1: PALETTE[0], 2: PALETTE[2], 3: PALETTE[1]}
for cls, grp in df_raw.groupby('pclass'):
    axes[0,1].scatter(grp['age'], grp['fare'].clip(upper=fare_p99),
        color=cls_colors[cls], alpha=0.4, s=35, label=f'Class {cls}')
axes[0,1].set_title('Age vs Fare (coloured by Class)')
axes[0,1].set_xlabel('Age'); axes[0,1].set_ylabel('Fare (£)')
axes[0,1].legend()

# Violin: fare per person vs survival
tmp = df.copy()
tmp['Outcome'] = df['survived'].map({0:'Died', 1:'Survived'})
sns.violinplot(data=tmp, x='Outcome', y='fare_per_person',
    ax=axes[0,2], palette=[PALETTE[1], PALETTE[2]], inner='box')
axes[0,2].set_title('Fare Per Person vs Survival')
axes[0,2].set_ylim(0, df['fare_per_person'].quantile(0.97))

# Grouped bar: survival by sex × class
sex_cls = df_raw.groupby(['pclass','sex'])['survived'].mean().unstack() * 100
x, w = np.arange(3), 0.35
axes[1,0].bar(x-w/2, sex_cls['female'], w, label='Female', color=PALETTE[4], edgecolor='white')
axes[1,0].bar(x+w/2, sex_cls['male'], w, label='Male', color=PALETTE[0], edgecolor='white')
axes[1,0].set_title('Survival Rate: Sex × Class')
axes[1,0].set_xticks(x); axes[1,0].set_xticklabels(['1st','2nd','3rd'])
axes[1,0].set_ylabel('Survival Rate (%)'); axes[1,0].legend()

# Family size survival (line)
fam_surv = df.groupby('family_size')['survived'].mean() * 100
axes[1,1].plot(fam_surv.index, fam_surv.values, 'o-',
    color=PALETTE[0], linewidth=2.5, markersize=8, markeredgecolor='white')
axes[1,1].fill_between(fam_surv.index, fam_surv.values, alpha=0.12, color=PALETTE[0])
axes[1,1].set_title('Survival Rate by Family Size')
axes[1,1].set_xlabel('Family Size'); axes[1,1].set_ylabel('Survival Rate (%)')

# KDE density: age by survival outcome
for surv, label, color in [(0,'Died',PALETTE[1]),(1,'Survived',PALETTE[2])]:
    df[df['survived']==surv]['age'].plot.kde(ax=axes[1,2], label=label, color=color, lw=2.5)
axes[1,2].set_title('Age Density: Survived vs Died')
axes[1,2].set_xlabel('Age'); axes[1,2].legend()
axes[1,2].set_xlim(0, 80)

plt.tight_layout()
plt.show()

### 3d. Pair Plot — Multi-Variable Relationships

In [ ]:
# Pair plot: pairwise scatter + KDE diagonal coloured by survival
pair_df = df[['survived','age','fare','family_size','fare_per_person']].copy()
pair_df['Survived'] = pair_df['survived'].map({0:'Died', 1:'Survived'})

g = sns.pairplot(
    pair_df.drop(columns='survived'),
    hue='Survived',
    palette={'Died': PALETTE[1], 'Survived': PALETTE[2]},
    diag_kind='kde',
    plot_kws={'alpha': 0.4, 'edgecolor': 'none', 's': 20},
    diag_kws={'linewidth': 2.0}
)
g.figure.suptitle('Pair Plot: Multi-Variable Relationships', fontsize=14, fontweight='bold', y=1.02)
plt.show()

### 3d. Correlation Analysis — Heatmap & Ranked Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8), facecolor=BG)
fig.suptitle('Correlation Analysis', fontsize=15, fontweight='bold')

corr_cols = ['survived','age','sibsp','parch','fare','family_size',
             'is_alone','sex_encoded','embarked_encoded','fare_per_person',
             'pclass_1','pclass_2','pclass_3']
corr = df[corr_cols].corr().round(3)

# Full heatmap (lower triangle)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    ax=axes[0], linewidths=0.5, linecolor='white', vmin=-1, vmax=1,
    cbar_kws={'shrink': 0.8}, annot_kws={'size': 8})
axes[0].set_title('Full Correlation Heatmap')
axes[0].tick_params(axis='x', rotation=35)

# Ranked correlation with survived
surv_corr = corr['survived'].drop('survived').sort_values()
colors_bar = [PALETTE[1] if v < 0 else PALETTE[2] for v in surv_corr.values]
axes[1].barh(surv_corr.index, surv_corr.values, color=colors_bar, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
for i, (idx, v) in enumerate(surv_corr.items()):
    axes[1].text(v + (0.01 if v >= 0 else -0.01), i, f'{v:+.3f}',
                 va='center', ha='left' if v >= 0 else 'right', fontsize=9)
axes[1].set_title('Correlation with Survived — Ranked')
axes[1].set_xlabel('Pearson Correlation Coefficient')

plt.tight_layout()
plt.show()

---
## Step 3e — Key Insights & Findings

In [ ]:
print(open('../outputs/key_insights.txt').read())

---
## Step 7 — Documentation

All methodology is documented in:
- `analysis.py` — inline code comments for every step
- `outputs/key_insights.txt` — structured findings
- `outputs/process_documentation.txt` — full methodology report
- `README.md` — project overview and setup guide

## Step 8 — Version Control

See `GIT_COMMANDS.txt` for the full step-by-step git workflow:
1. `git init`
2. `git add .` + `git commit -m "Initial commit"`
3. Create GitHub repo at https://github.com/new
4. `git remote add origin <url>` + `git push -u origin main`